### Full Startup Sequence

In [1]:
# Start Docker infrastructure (Cassandra, Kafka, Zookeeper)
docker-compose up -d

# Wait for Cassandra cluster formation
docker exec cassandra-1 nodetool status

# Start producer
python3 producer.py

# Start Spark consumer
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,com.datastax.spark:spark-cassandra-connector_2.12:3.5.0 \
  --conf spark.cassandra.connection.host=localhost \
  --conf spark.cassandra.connection.port=9042 \
  spark_consumer.py

SyntaxError: invalid syntax (4036559506.py, line 2)

### Updated startup sequence (enables avro data contracts)

In [ ]:
# 1. Infrastructure
docker-compose up -d zookeeper kafka schema-registry

# 2. Wait ~15 seconds for Schema Registry to be ready, then verify
curl http://localhost:8081/subjects         # should return []

# 3. Cassandra cluster (slow to start — wait for all 3 nodes)
docker-compose up -d cassandra-1
# wait for healthy, then:
docker-compose up -d cassandra-2 cassandra-3

# 4. Start producer first — this registers the schema in the Registry
python producer.py

# 5. In a separate terminal, start the consumer

spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,\
com.datastax.spark:spark-cassandra-connector_2.12:3.5.0,\
org.apache.spark:spark-avro_2.12:3.5.0 \
  --conf spark.cassandra.connection.host=localhost \
  --conf spark.cassandra.connection.port=9042 \
  spark_consumer.py

## Cassandra Main Features

### The Masterless Architecture

Unlike traditional Master-Slave databases, Cassandra is **decentralized**.

* **Peer-to-Peer:** Every node is equal. Any node can be a **Coordinator** for a request.
* **Gossip Protocol:** Nodes "talk" to each other to share health and state information. No central monitor or ZooKeeper required.
* **Linear Scalability:** To handle more traffic, simply add more nodes to the ring.

<center>
<img src="cassandra_masterless.png" style="width:700px;"/>
</center>

### Distributed Hash Ring + Virtual Nodes

Data is distributed across the cluster using **Consistent Hashing**.

* **Partition Key:** Cassandra hashes this key to a 160-bit integer (token).
* **Virtual Nodes (VNodes):** Each physical node owns multiple small token ranges.
* **Replication:** Data is replicated **clockwise** around the ring based on the **Replication Factor (RF)**.

<center>
<img src="cassandra-vnodes.png" style="width:600px;"/>
</center>

In [1]:
!docker ps -q | xargs -n 1 docker inspect -f '{{{{ .Name }}}} - {{{{range .NetworkSettings.Networks}}}}{{{{ .IPAddress }}}}{{{{end}}}}' | sed 's/\///'

cassandra-3 - 172.19.0.5
cassandra-2 - 172.19.0.6
kafka - 172.19.0.4
cassandra-1 - 172.19.0.3
zookeeper - 172.19.0.2


In [13]:
!docker exec cassandra-1 nodetool status

Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.19.0.6  1.02 MiB    16      74.7%             e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1
UN  172.19.0.5  1.07 MiB    16      77.4%             6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.19.0.7  788.41 KiB  16      73.2%             923bcc33-f9d8-4708-9e9c-581719649f2f  rack1
UN  172.19.0.3  1.01 MiB    16      74.7%             c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1



#### Virtual Nodes

In [14]:
!docker exec cassandra-1 nodetool ring | grep "^[0-9]" | sort -V | cat -n

     1	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              557021932782416796                          
     2	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              1292937625493310437                         
     3	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              2372310432940912095                         
     4	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              3615830438621504067                         
     5	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              4583194681700039566                         
     6	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              6173199234213938204                         
     7	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              7792222529910981501                         
     8	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%   

### How does distribution change if we add 4th node?

**Objective**: Show how Cassandra automatically rebalances data when a new node joins the cluster using virtual nodes (vnodes).

**Key Concepts**:
- **Token Ring**: Data distributed across nodes based on hash tokens (-2^63 to 2^63-1)
- **Virtual Nodes (vnodes)**: Each physical node owns multiple token ranges (16 vnodes per node in our setup)
- **Automatic Rebalancing**: New node takes ~25% of data from existing nodes without manual intervention

#### Cluster Status (BEFORE)

In [4]:
%%bash
echo "=== Current 3-Node Cluster Status ==="
docker exec cassandra-1 nodetool status

echo ""
echo "=== Token Ownership (iot_analytics keyspace) ==="
docker exec cassandra-1 nodetool status iot_analytics

=== Current 3-Node Cluster Status ===
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.19.0.6  356.73 KiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1
UN  172.19.0.5  350.68 KiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.19.0.3  344.67 KiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1


=== Token Ownership (iot_analytics keyspace) ===
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.19.0.6  356.73 KiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1
UN  172.19.0.5  350.68 KiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.19.0.3  344.67 KiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149

Check which node hold the data (AFTER)

In [12]:
!docker exec cassandra-1 nodetool getendpoints iot_analytics sensor_events ad618d2c-192c-4bed-bdff-cb3c33cdc7c5

172.19.0.6
172.19.0.7
172.19.0.5


#### New Node Creation

In [6]:
!docker run --name cassandra-4 \
  --network iot-cassandra-pipeline_iot-network \
  -m 1g \
  -e CASSANDRA_SEEDS=cassandra-1 \
  -e CASSANDRA_CLUSTER_NAME="IoT-Cluster" \
  -e CASSANDRA_DC=dc1 \
  -e CASSANDRA_RACK=rack1 \
  -e CASSANDRA_ENDPOINT_SNITCH=GossipingPropertyFileSnitch \
  -e MAX_HEAP_SIZE="512M" \
  -e HEAP_NEWSIZE="100M" \
  -d cassandra:4.1

f48228875d7214be3eb82148c29e0d01330d5d73ad02f9ba0d31a5e7bc2b2dcc


In [8]:
!docker ps -a

CONTAINER ID   IMAGE                             COMMAND                  CREATED         STATUS                   PORTS                                                                            NAMES
f48228875d72   cassandra:4.1                     "docker-entrypoint.s…"   8 seconds ago   Up 7 seconds             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-4
57979ca0581a   cassandra:4.1                     "docker-entrypoint.s…"   4 minutes ago   Up 4 minutes             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-3
8638950f4142   cassandra:4.1                     "docker-entrypoint.s…"   4 minutes ago   Up 4 minutes             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-2
e55f7c8d2cbc   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   4 minutes ago   Up 4 minutes             0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp                

#### Cluster Status (AFTER)

In [11]:
%%bash
echo "=== Current Cluster Status ==="
docker exec cassandra-1 nodetool status

echo ""
echo "=== Token Ownership (iot_analytics keyspace) ==="
docker exec cassandra-1 nodetool status iot_analytics

=== Current Cluster Status ===
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.19.0.6  356.73 KiB  16      74.7%             e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1
UN  172.19.0.5  1.07 MiB    16      77.4%             6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.19.0.7  842.18 KiB  16      73.2%             923bcc33-f9d8-4708-9e9c-581719649f2f  rack1
UN  172.19.0.3  1.01 MiB    16      74.7%             c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1


=== Token Ownership (iot_analytics keyspace) ===
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.19.0.6  356.73 KiB  16      74.7%             e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1
UN  172.19.0.5  1.07 MiB    16      77.4%             6d110fa3-3de5-404d-b6a6-6c31e392628f  

#### Check which node hold the data (AFTER)

In [15]:
!docker exec cassandra-1 nodetool getendpoints iot_analytics sensor_events d618d2c1-192c-4bed-bdff-cb3c33cdc7c5

172.19.0.5
172.19.0.3
172.19.0.6


### Check new vnodes distribution

In [16]:
!docker exec cassandra-1 nodetool ring | grep "^[0-9]" | sort -V | cat -n

     1	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              557021932782416796                          
     2	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              1292937625493310437                         
     3	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              2372310432940912095                         
     4	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              3615830438621504067                         
     5	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              4583194681700039566                         
     6	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              6173199234213938204                         
     7	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%              7792222529910981501                         
     8	172.19.0.3       rack1       Up     Normal  1.01 MiB        74.69%   

#### Revert Cluster

In [13]:
!docker exec cassandra-4 nodetool decommission

In [14]:
!docker rm -f cassandra-4

cassandra-4


## Tunable Consistency Levels

#### The Consistency Levels vs. CAP

Cassandra allows you to choose between **Availability** and **Consistency** on a **per-query basis**.

| Level | CAP Classification | Behavior during a Network Partition |
| --- | --- | --- |
| **ANY** | **Pure AP** | **Write-only**. Allows the write to succeed even if all replica nodes are down (stores a "hint" on a coordinator). Zero read consistency. |
| **ONE** | **AP** | **High Availability**. The query succeeds if at least one replica is alive. Risks returning stale data if that node wasn't part of the last write. |
| **QUORUM** | **Balanced (CP-leaning)** | **Majority Rule**. Sacrifice availability (requires $RF/2 + 1$ nodes). Provides "Strong Consistency" when used for both Reads and Writes ($W + R > RF$). |
| **ALL** | **Pure CP** | **Strict Consistency**. If a single replica node is unreachable due to a partition, the entire operation fails. Highest data integrity, lowest fault tolerance. |
| **LOCAL_*** | **Regional AP** | **DC-Aware**. Isolates the consistency check to the local Data Center. Prevents cross-region latency/partitions from failing the query. |

To ensure you always read the most recent data, your settings must satisfy:

$$\Large W + R > RF$$

*(W = Write Consistency, R = Read Consistency, and RF = Replication Factor)*

### Test Cassandra Latency

In [ ]:
import time
from cassandra.cluster import Cluster
from cassandra import ConsistencyLevel

cluster = Cluster(['127.0.0.1'])
session = cluster.connect('iot_analytics')

def test_heavy_latency(cl_name, cl_value):
    session.default_consistency_level = cl_value
    start = time.perf_counter()
    
    # Using a real query
    session.execute("SELECT * FROM sensor_events WHERE device_id = ad618d2c-192c-4bed-bdff-cb3c33cdc7c5 LIMIT 1;")
    
    end = time.perf_counter()
    print(f"Consistency {cl_name}: {(end - start) * 1000:.2f} ms")

def test_heavy_latency(cl_name, cl_value):
    session.default_consistency_level = cl_value
    start = time.perf_counter()

    heavy_query = "SELECT * FROM sensor_events LIMIT 1000"
    
    # Fetching many rows forces data coordination
    result = session.execute(heavy_query)
    # Iterate to ensure the driver actually fetches all pages
    list(result) 
    
    end = time.perf_counter()
    print(f"Heavy Query {cl_name}: {(end - start) * 1000:.2f} ms")

test_heavy_latency("ONE", ConsistencyLevel.ONE)
test_heavy_latency("QUORUM", ConsistencyLevel.QUORUM)
test_heavy_latency("ALL", ConsistencyLevel.ALL)

Heavy Query ONE: 33.55 ms
Heavy Query QUORUM: 45.69 ms
Heavy Query ALL: 114.34 ms


## 4. High-Performance Storage (LSM-Trees)

Cassandra is optimized for **high-velocity writes** (perfect for IoT):

1. **Commit Log:** Immediate sequential write to disk for durability.
2. **Memtable:** Data sits in memory for lightning-fast access.
3. **SSTable:** Periodically, data is flushed to **immutable** files on disk.

**Sequential vs Random:** Cassandra avoids slow "random seeks" on disk, making it 10x–100x faster for writes than traditional SQL.

In [26]:
!docker exec cassandra-1 nodetool tablestats iot_analytics.sensor_events | grep -E "SSTable count|Space used \(live\)|Number of partitions"

		SSTable count: 3
		Old SSTable count: 0
		Space used (live): 28011298
		Number of partitions (estimate): 1000
